# NNLM(Neural Network Language Model)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim


In [9]:
# 컨텍스트 단어들을 임베딩 -> MLP로 변환해서 다음 단어 분포를 예측하는 모델
class NNLM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, context_size):
        super(NNLM, self).__init__()            # nn.Module 초기화
        self.embed = nn.Embedding(vocab_size, embed_size)   # 단어 ID -> 임베딩 벡터
        self.fc1 = nn.Linear(context_size * embed_size, hidden_size)    # (컨텍스트 *임베딩 연결) -> 은닉층
        self.relu = nn.ReLU()       # 비선형 추가
        self.fc2 = nn.Linear(hidden_size, vocab_size)   # 은닉층 -> 단어 분포 logit(어휘 크기)
        self.log_softmax = nn.LogSoftmax(dim = 1)       # logit -> 로그확률(배치기준 dim= 1 => 배치별 확률)
        
    def forward(self, x):
        embeds = self.embed(x)                      # (B, context_size) -> (B, context_size, embed_size)
        embeds = embeds.view(embeds.size(0), -1)    # (B, context_size * embed_size)로 컨텍스트 *임베딩 연결  
        output = self.fc1(embeds)       # 은닉층 선형 변환
        output = self.relu(output)      # ReLU 로 비선형 추가
        output = self.fc2(output)       # 은닉층 -> 어휘 크기만큼 로짓 출력
        log_probs = self.log_softmax(output)    # (B, vocab_size) 로 로그 확률 반환
        return log_probs
        

In [11]:
# 하이퍼파라미터 설정
VOCAB_SIZE = 5000
EMBED_SIZE = 300
HIDDEN_SIZE = 128
CONTEXT_SIZE = 2

model = NNLM(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, CONTEXT_SIZE)
model

NNLM(
  (embed): Embedding(5000, 300)
  (fc1): Linear(in_features=600, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=5000, bias=True)
  (log_softmax): LogSoftmax(dim=1)
)

In [20]:
# 더미 데이터 생성
X = torch.randint(0, VOCAB_SIZE, (8, CONTEXT_SIZE)) # 배치 8, 컨덱스트 2 범위로 단어 ID 랜덤 생성
y = torch.randint(0, VOCAB_SIZE, (8, ))             # 배치 8 다음단어 ID 랜덤 생성

X.shape, y.shape

(torch.Size([8, 2]), torch.Size([8]))

In [21]:
criterion = nn.NLLLoss()        # Negative Log Likelihood 손실함수: 로그확률용(log_softmax 출력용) 손실함수
optimizer = optim.Adam(model.parameters(), lr= 0.001)

model.train()
optimizer.zero_grad()
output = model(X)
loss = criterion(output, y)
loss.backward()
optimizer.step()

loss.item()

8.586114883422852